<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# 03: Preparing the data set
----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

We now look at the dataset and try to identify errors in the set.

First, load some `python` libraries and import the `iris data set` as `pandas` data frame `df`:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

Note, that the data set does not have a header line. We therefor manually add a header for
easier identificaion later...

In [2]:
df = pd.read_csv('data/iris_dirty.csv',
                 header=None,
                 encoding='iso-8859-15',
                 names=['sepal length', 'sepal width', 'petal length', 'petal width', 'class'])

Next, type first 10 lines:

In [5]:
df.head(10)

,sepal length,sepal width,petal length,petal width,class
0,5.1,3.5,1.4,2 mm,Iris-setosa
1,4.9,3.0,1.4,2 mm,Iris-setosa
2,4.7,3.2,1.3,2 mm,Iris-setosa
3,4.6,3.1,1.5,2 mm,Iris-setosa
4,5.0,3.6,1.4,2 mm,Iris-setosa
5,5.4,3.9,1.7,4 mm,Iris-setosa
6,4.6,3.4,1.4,3 mm,Iris-setosa
7,5.0,3.4,1.5,2 mm,Iris-setosa
8,4.4,2.9,1.4,2 mm,Iris-setosa
9,4.9,3.1,1.5,1 mm,Iris-setosa


It seems that sepal length, sepal width, and petal length are in **cm**, but the petal width in **mm**,
with the label added (Needs to be changed later).

### Identify and then fill missing values

Next, check the number of lines in the data set: 

In [6]:
df.count()

sepal length    151
sepal width     150
petal length    151
petal width     151
class           151
dtype: int64

We have 3 times 50 records, plus the header, makes 101 lines. Seems, that for the **sepal width**, an entry
is missing.

Check, which record of the **sepal width** has no number in:

In [ ]:
#df['sepal width'].isnull()
df[df['sepal width'].isnull()]

It is record line 82, an **iris versicolor**.

We grab all iris versicolor entries, and there the sepal width column, and calculate the **mean**:

In [ ]:
meanSepalWidth=pd.Series.mean(df[df['class']  == 'Iris-versicolor']['sepal width'])
print(meanSepalWidth)

Then we replace the empty values (NaN) in line 82 with the mean value:

In [ ]:
df.loc[82]

In [ ]:
df.loc[82,'sepal width'] = meanSepalWidth

In [ ]:
df.loc[82]

... done

### Identify double lines

Next, we search for double entries, and we find 5 of them. Of couse, this can be ok, measurements can
be the same...

In [ ]:
df[df.duplicated(keep=False)]

... but then we check in each iris group, how many entries we have. We should have 50!

In [ ]:
df.groupby('class').count()

Obviously, iris versicolor has one entry too much, the line 100 identified before. Remove and check:

In [ ]:
df = df.drop(df.index[[100]])

In [ ]:
df.groupby('class').count()

... done

### Correct typo for class

As a side result, we found two entries for iris setosa, one with the typo **iris setsoa**.

We locate the line  and replac ethe label...

In [ ]:
df[df['class']  == 'Iris-setsoa']

In [ ]:
df.loc[49,'class'] = 'Iris-setosa'

In [ ]:
df.groupby('class').count()

... done

### Change petal width to cm

Next, we homogenize the data by converting the petal width to cm too, and removing the *mm* unit.

We define a small function for this task:

In [ ]:
pd.to_numeric('2 mm'.replace(' mm', '')) / 10

In [ ]:
def convert_from_mm(row):
    return pd.to_numeric(row['petal width'].replace(' mm', '')) / 10

In [ ]:
df['petal width'] = df.apply(convert_from_mm, axis='columns')

In [ ]:
df.head()

... done

### Check for implausible numbers and correct

Finally, we plot some statistical information of each column to find out possible **outliers**.

There is a large value as maximum in the sepal length column.

In [ ]:
df.describe()

We can identify this outliers graphically in a **histogram plot**:

In [ ]:
df.hist(figsize=(10, 10))
plt.show()

In [ ]:
sns.jointplot(df['sepal length'], df['petal length'])
plt.show()

We identity the line, it is line 58, and correct this obvious typo.

In [ ]:
df[df['sepal length'] == 58]

In [ ]:
df.loc[143,'sepal length'] = 5.8

In [ ]:
sns.jointplot(df['sepal length'], df['petal length'])
plt.show()

That's it we are almost done. We need to save our cleaned database as a final step:

In [ ]:
df.to_csv('data/iris_cleaned.csv', index=False, header=True)

... done

----

[next>](03_iris3.ipynb)